# 01 — BIDMC-shaped respiratory states

This deterministic respiratory-like fixture is not a BIDMC record and does not
detect breaths. We preserve the raw signal, declare a rate, and compile rising,
falling, and inactive states.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import featuregraph as fg
from featuregraph.operators.states import rising_state

values = [0, .5, 1, 1, .5, 0, 0, .6, 1.2, .6, 0, 0]
observations = pd.DataFrame({
    "subject": "tutorial-subject",
    "sample": np.arange(len(values)),
    "respiration_raw": values,
})
observations["respiration_analysis"] = observations["respiration_raw"]
observations["rate"] = observations["respiration_analysis"].diff().fillna(0.0)


## Declare before executing

No smoothing is used. The first rate is explicitly set to zero because no
preceding observation exists; that is a recording-boundary policy, not a
physiological conclusion.


In [ ]:
rate = {"column": "rate"}
eps = {"parameter": "eps"}
contract = {
    "version": "state-contract-v1",
    "parameters": {"eps": 1e-12},
    "group_by": "subject",
    "states": {
        "rising": {"op": "gt", "left": rate, "right": eps},
        "falling": {"op": "lt", "left": rate,
                    "right": {"op": "neg", "value": eps}},
        "inactive": {"op": "le",
                     "left": {"op": "abs", "value": rate},
                     "right": eps},
    },
    "events": {
        "enter_rising": {"type": "enter_state", "state": "rising"},
        "exit_rising": {"type": "exit_state", "state": "rising"},
    },
}
compiled = fg.compile_states(observations, contract)
compiled.observations


In [ ]:
rising = fg.transition.Transition(
    observations.copy(), "respiration_analysis", "rising",
    rising_state, eps=1e-12,
)
rising.df


In [ ]:
frame = compiled.observations
colors = {"rising": "#2a9d8f", "falling": "#e76f51", "inactive": "#8d99ae"}
fig, ax = plt.subplots(figsize=(9, 4))
for state, part in frame.groupby("state", sort=False):
    ax.scatter(part["sample"], part["respiration_raw"], label=state,
               color=colors[state], s=60)
ax.plot(frame["sample"], frame["respiration_raw"], color="black", alpha=.4)
ax.legend(); ax.grid(alpha=.2); plt.show()


In [ ]:
expected = [
    "inactive", "rising", "rising", "inactive", "falling", "falling",
    "inactive", "rising", "rising", "falling", "falling", "inactive",
]
assert frame["state"].tolist() == expected
assert compiled.validation_report["passed"].all()
assert frame.loc[frame["enter_rising"], "sample"].tolist() == [1, 7]


The output locates states in this declared representation.
It does not establish breath truth, respiratory physiology, or clinical
validity. The maintained BIDMC study adds smoothing, tolerance,
plateau-aware objects, comparisons, and 53-record validation.
